In [ ]:
!pip install xgboost scikit-learn pandas joblib

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("/content/sri_lankan_food_dataset - Sri Lankan Foods Dataset.csv")

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# ============================================================
# HANDLE MISSING VALUES
# ============================================================

df = df.fillna("Unknown")

# ============================================================
# SHOW DATASET INFO
# ============================================================

print("Columns:")
print(df.columns)

print("\nDataset Preview:")
print(df.head())

# ============================================================
# SAVE ORIGINAL CATEGORY LABELS
# ============================================================

original_categories = df['category'].copy()

# ============================================================
# FEATURE COLUMNS
# ============================================================

feature_columns = [
    'category',
    'is_veg',
    'meal_time',
    'spicy_level',
    'price_range'
]

# ============================================================
# ENCODE FEATURE COLUMNS
# ============================================================

feature_encoders = {}

for col in feature_columns:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    feature_encoders[col] = le

# ============================================================
# VERIFY FEATURE TYPES
# ============================================================

print("\nFeature Data Types:")
print(df[feature_columns].dtypes)

# ============================================================
# CREATE FEATURES
# ============================================================

X = df[feature_columns].astype(int)

# ============================================================
# CREATE TARGET LABELS
# ============================================================

y = original_categories.astype(str)

# ============================================================
# TARGET LABELS
# ============================================================

y = original_categories.astype(str)

# ============================================================
# FIRST LABEL ENCODING
# ============================================================

initial_encoder = LabelEncoder()

y_encoded = initial_encoder.fit_transform(y)

# ============================================================
# REMOVE RARE CLASSES
# ============================================================

category_counts = pd.Series(y_encoded).value_counts()

valid_categories = category_counts[
    category_counts >= 2
].index

mask = pd.Series(y_encoded).isin(valid_categories)

X = X[mask]

y_encoded = pd.Series(y_encoded)[mask]

# ============================================================
# RE-ENCODE LABELS AGAIN
# VERY IMPORTANT
# ============================================================

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y_encoded)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)
# ============================================================
# CREATE XGBOOST MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softmax',
    eval_metric='mlogloss'
)

# ============================================================
# TRAIN MODEL
# ============================================================

model.fit(X_train, y_train)

# ============================================================
# MAKE PREDICTIONS
# ============================================================

preds = model.predict(X_test)

# ============================================================
# EVALUATE MODEL
# ============================================================

accuracy = accuracy_score(y_test, preds)

print(f"\nModel Accuracy: {accuracy:.2f}")

Columns:
Index(['name', 'category', 'is_veg', 'meal_time', 'spicy_level',
       'price_range'],
      dtype='object')

Dataset Preview:
             name    category  is_veg         meal_time spicy_level  \
0  String Hoppers  Main Meals    True  Breakfast/Dinner     Unknown   
1   Plain Hoppers  Main Meals    True  Breakfast/Dinner     Unknown   
2     Egg Hoppers  Main Meals   False  Breakfast/Dinner         Low   
3           Pittu  Main Meals    True         Breakfast     Unknown   
4        Kiribath  Main Meals    True         Breakfast         Low   

  price_range  
0         Low  
1         Low  
2         Low  
3         Low  
4         Low  

Feature Data Types:
category       int64
is_veg         int64
meal_time      int64
spicy_level    int64
price_range    int64
dtype: object

Model Accuracy: 0.97


In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

joblib.dump(model, "sri_lankan_food_model.pkl")

joblib.dump(feature_encoders, "feature_encoders.pkl")

joblib.dump(target_encoder, "target_encoder.pkl")

print("\nModel Saved Successfully!")


Model Saved Successfully!


In [ ]:
# ============================================================
# TEST MODEL WITH TOURIST PREFERENCES
# ============================================================

sample = {
    'category': 'Main Meals',
    'is_veg': 'True',
    'meal_time': 'Breakfast',
    'spicy_level': 'Low',
    'price_range': 'Low'
}

print("\nTourist Preferences:")
print(sample)

# ============================================================
# ENCODE INPUT
# ============================================================

input_data = []

for col in feature_columns:

    encoded_value = feature_encoders[col].transform(
        [str(sample[col])]
    )[0]

    input_data.append(encoded_value)

# ============================================================
# CREATE INPUT DATAFRAME
# ============================================================

input_df = pd.DataFrame(
    [input_data],
    columns=feature_columns
).astype(int)

# ============================================================
# MAKE PREDICTION
# ============================================================

prediction = model.predict(input_df)

# ============================================================
# DECODE PREDICTION
# ============================================================

predicted_category_encoded = prediction[0]

predicted_category = feature_encoders[
    'category'
].inverse_transform(
    [predicted_category_encoded]
)

print("\nPredicted Food Category:")
print(predicted_category[0])

# ============================================================
# FIND MATCHING FOODS
# ============================================================

recommended_foods = df[
    df['category'] == predicted_category_encoded
]

# ============================================================
# SHOW RECOMMENDATIONS
# ============================================================

print("\nRecommended Foods:")

print(
    recommended_foods[
        [
            'name',
            'meal_time',
            'spicy_level',
            'price_range'
        ]
    ]
    .drop_duplicates()
    .head(10)
)


Tourist Preferences:
{'category': 'Main Meals', 'is_veg': 'True', 'meal_time': 'Breakfast', 'spicy_level': 'Low', 'price_range': 'Low'}

Predicted Food Category:
Main Meals

Recommended Foods:
                    name  meal_time  spicy_level  price_range
0         String Hoppers          2            3            1
1          Plain Hoppers          2            3            1
2            Egg Hoppers          2            1            1
3                  Pittu          1            3            1
4               Kiribath          1            1            1
5     Kadala (Chickpeas)          1            1            1
6          Boiled Manioc          1            3            1
7  Boiled Sweet Potatoes          1            3            1
8                   Dosa          2            3            1
9               Pol Roti          2            3            1
